# **Import**

In [ ]:
import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import mean_absolute_error, mean_squared_error

from lightgbm import LGBMRegressor

# **Data Load**

In [ ]:
cd /content/drive/MyDrive/[Projects]/Dacon/제3회 국민대학교 AI빅데이터 분석 경진대회/Data

/content/drive/MyDrive/[Projects]/Dacon/제3회 국민대학교 AI빅데이터 분석 경진대회/Data


In [ ]:
train_df = pd.read_csv('./train.csv')
pair_df = pd.read_csv('./pairs.csv')

In [ ]:
train_df

,item_id,year,month,seq,type,hs4,weight,quantity,value
0,DEWLVASR,2022,1,1.0,1,3038,14858.0,0.0,32688.0
1,ELQGMQWE,2022,1,1.0,1,2002,62195.0,0.0,110617.0
2,AHMDUILJ,2022,1,1.0,1,2102,18426.0,0.0,72766.0
3,XIPPENFQ,2022,1,1.0,1,2501,20426.0,0.0,11172.0
4,FTSVTTSR,2022,1,1.0,1,2529,248000.0,0.0,143004.0
...,...,...,...,...,...,...,...,...,...
10831,XIFHSOWQ,2025,7,3.0,1,8708,352.0,0.0,12937.0
10832,FITUEHWN,2025,7,3.0,1,8714,655.0,900.0,16054.0
10833,UGEQLMXM,2025,7,3.0,1,8714,758.0,0.0,74377.0
10834,BLANHGYY,2025,7,3.0,1,9022,345.0,2.0,69720.0


In [ ]:
pair_df

,leading_item_id,following_item_id,best_lag,max_corr
0,AANGBULD,APQGTRMF,5,-0.443984
1,AANGBULD,DEWLVASR,6,0.640221
2,AANGBULD,DNMPSKTB,4,-0.410635
3,AANGBULD,EVBVXETX,6,0.436623
4,AANGBULD,FTSVTTSR,3,0.531400
...,...,...,...,...
1420,ZXERAXWP,DBWLZWNK,6,-0.470150
1421,ZXERAXWP,FITUEHWN,4,0.406369
1422,ZXERAXWP,MIRCVAMV,3,-0.435495
1423,ZXERAXWP,UIFPPCLR,1,-0.526205


# **Features for Model Input**

In [ ]:
monthly = (
    train_df
    .groupby(['item_id', 'year', 'month'], as_index=False)['value']
    .sum()
)
monthly['ym'] = pd.to_datetime(monthly['year'].astype(str) + '-' + monthly['month'].astype(str))
pivot = monthly.pivot(index='item_id', columns='ym', values='value')
pivot = pivot.fillna(0)
pivot

ym,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01
item_id,,,,,,,,,,,,,,,,,,,,,
AANGBULD,14276.0,52347.0,53549.0,0.0,26997.0,84489.0,0.0,0.0,0.0,0.0,...,428725.0,144248.0,26507.0,25691.0,25805.0,0.0,38441.0,0.0,441275.0,533478.0
AHMDUILJ,242705.0,120847.0,197317.0,126142.0,71730.0,149138.0,186617.0,169995.0,140547.0,89292.0,...,123085.0,143451.0,78649.0,125098.0,80404.0,157401.0,115509.0,127473.0,89479.0,101317.0
ANWUJOKX,0.0,0.0,0.0,63580.0,81670.0,26424.0,8470.0,0.0,0.0,80475.0,...,0.0,0.0,0.0,27980.0,0.0,0.0,0.0,0.0,0.0,0.0
APQGTRMF,383999.0,512813.0,217064.0,470398.0,539873.0,582317.0,759980.0,216019.0,537693.0,205326.0,...,683581.0,2147.0,0.0,25013.0,77.0,20741.0,2403.0,3543.0,32430.0,40608.0
ATLDMDBO,143097177.0,103568323.0,118403737.0,121873741.0,115024617.0,65716075.0,146216818.0,97552978.0,72341427.0,87454167.0,...,60276050.0,30160198.0,42613728.0,64451013.0,38667429.0,29354408.0,42450439.0,37136720.0,32181798.0,57090235.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YSYHGLQK,0.0,543.0,766.0,1108.0,859.0,1426.0,2413.0,638.0,0.0,1199.0,...,188.0,541.0,696.0,8710.0,3175.0,2624.0,0.0,182.0,2128.0,10651.0
ZCELVYQU,373859.0,59900.0,31158.0,594407.0,648232.0,496737.0,210179.0,0.0,70748.0,15512.0,...,0.0,609803.0,23712.0,654630.0,4496.0,1177300.0,1187539.0,26434.0,115631.0,270262.0
ZGJXVMNI,1154724.0,1337622.0,1662893.0,1561647.0,1603223.0,1641942.0,1815161.0,1546959.0,1536799.0,1496906.0,...,3168505.0,3059865.0,1579976.0,1413293.0,3038078.0,2915914.0,3565526.0,3020051.0,2412781.0,2458481.0


In [ ]:
max_self_lag = 2
ma_window = 3

all_rows = []
for _, row in tqdm(pair_df.iterrows(), total=len(pair_df)):
    leader = row['leading_item_id']
    follower = row['following_item_id']
    best_lag = int(row['best_lag'])

    s_lead = pivot.loc[leader]
    s_follow = pivot.loc[follower]

    for ym in pivot.columns:
        try:
            leader_lag_val = s_lead[ym - pd.DateOffset(months=best_lag)]
            follower_lags = {f'follower_lag{l}': s_follow[ym - pd.DateOffset(months=l)] for l in range(1, max_self_lag + 1)}
            leader_ma = s_lead[:ym - pd.DateOffset(months=1)].rolling(ma_window).mean().iloc[-1]
            follower_ma = s_follow[:ym - pd.DateOffset(months=1)].rolling(ma_window).mean().iloc[-1]
            target = s_follow[ym]
        except Exception:
            continue

        rec = {
            'month': ym,
            'leader_item': leader,
            'follower_item': follower,
            'leader_value_lag': leader_lag_val,
            'leader_ma': leader_ma,
            'follower_ma': follower_ma,
            'target': target
        }
        rec.update(follower_lags)
        all_rows.append(rec)

    feature_df = pd.DataFrame(all_rows)
    feature_df = feature_df.sort_values(['leader_item', 'follower_item', 'month']).reset_index(drop=True)

feature_df.head()

100%|██████████| 1425/1425 [03:35<00:00,  6.60it/s]


,month,leader_item,follower_item,leader_value_lag,leader_ma,follower_ma,target,follower_lag1,follower_lag2
0,2022-06-01,AANGBULD,APQGTRMF,14276.0,26848.666667,409111.666667,582317.0,539873.0,470398.0
1,2022-07-01,AANGBULD,APQGTRMF,52347.0,37162.000000,530862.666667,759980.0,582317.0,539873.0
2,2022-08-01,AANGBULD,APQGTRMF,53549.0,37162.000000,627390.000000,216019.0,759980.0,582317.0
3,2022-09-01,AANGBULD,APQGTRMF,0.0,28163.000000,519438.666667,537693.0,216019.0,759980.0
4,2022-10-01,AANGBULD,APQGTRMF,26997.0,0.000000,504564.000000,205326.0,537693.0,216019.0


In [ ]:
feature_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55954 entries, 0 to 55953
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   month             55954 non-null  datetime64[ns]
 1   leader_item       55954 non-null  object        
 2   follower_item     55954 non-null  object        
 3   leader_value_lag  55954 non-null  float64       
 4   leader_ma         55492 non-null  float64       
 5   follower_ma       55492 non-null  float64       
 6   target            55954 non-null  float64       
 7   follower_lag1     55954 non-null  float64       
 8   follower_lag2     55954 non-null  float64       
dtypes: datetime64[ns](1), float64(6), object(2)
memory usage: 3.8+ MB


In [ ]:
feature_df['target'].describe()

,target
count,5.595400e+04
mean,5.076061e+06
std,1.385521e+07
min,0.000000e+00
25%,7.840500e+04
50%,4.901070e+05
75%,4.236970e+06
max,1.462168e+08


In [ ]:
train_cutoff = pd.Timestamp('2025-07-01') - pd.DateOffset(months=1)
val_month = pd.Timestamp('2025-07-01')

train_df_features = feature_df[feature_df['month'] <= train_cutoff].copy()
val_df_features = feature_df[feature_df['month'] == val_month].copy()

x_cols = ['leader_value_lag', 'follower_lag1', 'follower_lag2', 'leader_ma', 'follower_ma']
train_x = train_df_features[x_cols]
train_y = train_df_features['target']
val_x = val_df_features[x_cols]
val_y = val_df_features['target']

In [ ]:
model = LGBMRegressor(n_estimators=1000, learning_rate=0.05)
model.fit(train_x, train_y, eval_set=[(val_x, val_y)], eval_metric='mae')

pred_y = model.predict(val_x)
print(f'MAE: {mean_absolute_error(val_y, pred_y)}')
print(f'RMSE: {mean_squared_error(val_y, pred_y)**0.5}')

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002769 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 54529, number of used features: 5
[LightGBM] [Info] Start training from score 5082103.550753
MAE: 1686181.2024241197
RMSE: 4705886.766227576


In [ ]:
max_self_lag = 12
ma_windows = [3, 6, 12]

all_rows = []
for _, row in tqdm(pair_df.iterrows(), total=len(pair_df)):
    leader = row['leading_item_id']
    follower = row['following_item_id']
    best_lag = int(row['best_lag'])

    s_lead = pivot.loc[leader]
    s_follow = pivot.loc[follower]

    for ym in pivot.columns:
        rec = {'month': ym, 'leader_item': leader, 'follower_item': follower}

        # Leader lag 주변 추가
        for lag_shift in [-1, 0, 1]:
            lag = best_lag + lag_shift
            if lag <= 0:
                continue
            rec[f'leader_lag{lag}'] = s_lead.get(ym - pd.DateOffset(months=lag), np.nan)

        # Follower lag 확장
        for l in range(1, max_self_lag+1):
            rec[f'follower_lag{l}'] = s_follow.get(ym - pd.DateOffset(months=l), np.nan)

        # 이동평균
        for w in ma_windows:
            rec[f'leader_ma{w}'] = s_lead[:ym - pd.DateOffset(months=1)].rolling(w).mean().iloc[-1] if len(s_lead[:ym - pd.DateOffset(months=1)]) >= w else np.nan
            rec[f'follower_ma{w}'] = s_follow[:ym - pd.DateOffset(months=1)].rolling(w).mean().iloc[-1] if len(s_follow[:ym - pd.DateOffset(months=1)]) >= w else np.nan

        # Month feature (계절성)
        rec['month_num'] = ym.month

        # Target
        rec['target'] = s_follow.get(ym, np.nan)

        all_rows.append(rec)

extended_feature_df = pd.DataFrame(all_rows)
extended_feature_df = extended_feature_df.sort_values(['leader_item', 'follower_item', 'month']).reset_index(drop=True)

extended_feature_df.head()

100%|██████████| 1425/1425 [04:11<00:00,  5.66it/s]


,month,leader_item,follower_item,leader_lag4,leader_lag5,leader_lag6,follower_lag1,follower_lag2,follower_lag3,follower_lag4,...,leader_ma6,follower_ma6,leader_ma12,follower_ma12,month_num,target,leader_lag7,leader_lag3,leader_lag2,leader_lag1
0,2022-01-01,AANGBULD,APQGTRMF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1,383999.0,NaN,NaN,NaN,NaN
1,2022-02-01,AANGBULD,APQGTRMF,NaN,NaN,NaN,383999.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2,512813.0,NaN,NaN,NaN,NaN
2,2022-03-01,AANGBULD,APQGTRMF,NaN,NaN,NaN,512813.0,383999.0,NaN,NaN,...,NaN,NaN,NaN,NaN,3,217064.0,NaN,NaN,NaN,NaN
3,2022-04-01,AANGBULD,APQGTRMF,NaN,NaN,NaN,217064.0,512813.0,383999.0,NaN,...,NaN,NaN,NaN,NaN,4,470398.0,NaN,NaN,NaN,NaN
4,2022-05-01,AANGBULD,APQGTRMF,14276.0,NaN,NaN,470398.0,217064.0,512813.0,383999.0,...,NaN,NaN,NaN,NaN,5,539873.0,NaN,NaN,NaN,NaN


In [ ]:
extended_feature_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61275 entries, 0 to 61274
Data columns (total 30 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   month           61275 non-null  datetime64[ns]
 1   leader_item     61275 non-null  object        
 2   follower_item   61275 non-null  object        
 3   leader_lag4     26949 non-null  float64       
 4   leader_lag5     27892 non-null  float64       
 5   leader_lag6     18574 non-null  float64       
 6   follower_lag1   59850 non-null  float64       
 7   follower_lag2   58425 non-null  float64       
 8   follower_lag3   57000 non-null  float64       
 9   follower_lag4   55575 non-null  float64       
 10  follower_lag5   54150 non-null  float64       
 11  follower_lag6   52725 non-null  float64       
 12  follower_lag7   51300 non-null  float64       
 13  follower_lag8   49875 non-null  float64       
 14  follower_lag9   48450 non-null  float64       
 15  fo

In [ ]:
extended_feature_df['target'].describe()

,target
count,6.127500e+04
mean,5.307416e+06
std,1.461316e+07
min,0.000000e+00
25%,7.726900e+04
50%,5.006040e+05
75%,4.374113e+06
max,1.462168e+08


In [ ]:
train_cutoff = pd.Timestamp('2025-07-01') - pd.DateOffset(months=1)
val_month = pd.Timestamp('2025-07-01')

train_df_features = extended_feature_df[extended_feature_df['month'] <= train_cutoff].copy()
val_df_features = extended_feature_df[extended_feature_df['month'] == val_month].copy()

x_cols = [col for col in extended_feature_df.columns
          if col.startswith('leader_lag')
          or col.startswith('follower_lag')
          or col.startswith('leader_ma')
          or col.startswith('follower_ma')]
train_x = train_df_features[x_cols]
train_y = train_df_features['target']
val_x = val_df_features[x_cols]
val_y = val_df_features['target']

In [ ]:
model = LGBMRegressor(n_estimators=1000, learning_rate=0.05)
model.fit(train_x, train_y, eval_set=[(val_x, val_y)], eval_metric='mae')

pred_y = model.predict(val_x)
print(f'MAE: {mean_absolute_error(val_y, pred_y)}')
print(f'RMSE: {mean_squared_error(val_y, pred_y)**0.5}')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004836 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6375
[LightGBM] [Info] Number of data points in the train set: 59850, number of used features: 25
[LightGBM] [Info] Start training from score 5318430.526583
MAE: 1702172.8453989096
RMSE: 4737897.773908735


In [ ]:
train_cutoff = pd.Timestamp('2025-07-01') - pd.DateOffset(months=1)
val_month = pd.Timestamp('2025-07-01')

train_df_features = extended_feature_df[extended_feature_df['month'] <= train_cutoff].copy()
val_df_features = extended_feature_df[extended_feature_df['month'] == val_month].copy()

x_cols = [
    'leader_lag1', 'leader_lag2', 'leader_lag3',
    'follower_lag1', 'follower_lag2', 'follower_lag3',
    'leader_ma3', 'leader_ma6',
    'follower_ma3', 'follower_ma6'
]
train_x = train_df_features[x_cols]
train_y = train_df_features['target']
val_x = val_df_features[x_cols]
val_y = val_df_features['target']

In [ ]:
model = LGBMRegressor(n_estimators=1000, learning_rate=0.05)
model.fit(train_x, train_y, eval_set=[(val_x, val_y)], eval_metric='mae')

pred_y = model.predict(val_x)
print(f'MAE: {mean_absolute_error(val_y, pred_y)}')
print(f'RMSE: {mean_squared_error(val_y, pred_y)**0.5}')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001929 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 59850, number of used features: 10
[LightGBM] [Info] Start training from score 5318430.526583
MAE: 1690142.1968267749
RMSE: 4590976.227064201


# **Prediction**

In [ ]:
pred_month = pd.Timestamp('2025-08-01')

all_rows = []
for _, row in pair_df.iterrows():
    leader = row['leading_item_id']
    follower = row['following_item_id']
    best_lag = int(row['best_lag'])

    s_lead = pivot.loc[leader]
    s_follow = pivot.loc[follower]

    rec = {'leader_item': leader, 'follower_item': follower, 'month': pred_month}

    # leader lag
    rec['leader_lag1'] = s_lead.get(pred_month - pd.DateOffset(months=1), np.nan)
    rec['leader_lag2'] = s_lead.get(pred_month - pd.DateOffset(months=2), np.nan)
    rec['leader_lag3'] = s_lead.get(pred_month - pd.DateOffset(months=3), np.nan)

    # follower lag
    rec['follower_lag1'] = s_follow.get(pred_month - pd.DateOffset(months=1), np.nan)
    rec['follower_lag2'] = s_follow.get(pred_month - pd.DateOffset(months=2), np.nan)
    rec['follower_lag3'] = s_follow.get(pred_month - pd.DateOffset(months=3), np.nan)

    # 이동평균
    rec['leader_ma3'] = s_lead[:pred_month - pd.DateOffset(months=1)].rolling(3).mean().iloc[-1]
    rec['leader_ma6'] = s_lead[:pred_month - pd.DateOffset(months=1)].rolling(6).mean().iloc[-1]
    rec['follower_ma3'] = s_follow[:pred_month - pd.DateOffset(months=1)].rolling(3).mean().iloc[-1]
    rec['follower_ma6'] = s_follow[:pred_month - pd.DateOffset(months=1)].rolling(6).mean().iloc[-1]

    all_rows.append(rec)

pred_df = pd.DataFrame(all_rows)
pred_df.head()

,leader_item,follower_item,month,leader_lag1,leader_lag2,leader_lag3,follower_lag1,follower_lag2,follower_lag3,leader_ma3,leader_ma6,follower_ma3,follower_ma6
0,AANGBULD,APQGTRMF,2025-08-01,533478.0,441275.0,0.0,40608.0,32430.0,3543.0,324917.666667,173166.5,2.552700e+04,1.663367e+04
1,AANGBULD,DEWLVASR,2025-08-01,533478.0,441275.0,0.0,482787.0,408887.0,361589.0,324917.666667,173166.5,4.177543e+05,4.393930e+05
2,AANGBULD,DNMPSKTB,2025-08-01,533478.0,441275.0,0.0,4507669.0,5676571.0,5318257.0,324917.666667,173166.5,5.167499e+06,5.344211e+06
3,AANGBULD,EVBVXETX,2025-08-01,533478.0,441275.0,0.0,5061099.0,5000609.0,5411733.0,324917.666667,173166.5,5.157814e+06,5.021246e+06
4,AANGBULD,FTSVTTSR,2025-08-01,533478.0,441275.0,0.0,246916.0,129389.0,134908.0,324917.666667,173166.5,1.704043e+05,1.763723e+05


In [ ]:
train_cutoff = pd.Timestamp('2025-07-01') - pd.DateOffset(months=1)
val_month = pd.Timestamp('2025-07-01')

train_df_features = extended_feature_df[extended_feature_df['month'] <= train_cutoff].copy()
val_df_features = extended_feature_df[extended_feature_df['month'] == val_month].copy()

train_df_features = train_df_features.dropna(subset=x_cols + ['target'])
val_df_features = val_df_features.dropna(subset=x_cols + ['target'])

train_x = train_df_features[x_cols]
train_y = train_df_features['target']
val_x = val_df_features[x_cols]
val_y = val_df_features['target']

In [ ]:
model = LGBMRegressor(n_estimators=1000, learning_rate=0.05)
model.fit(train_x, train_y, eval_set=[(val_x, val_y)])

# 4) 8월 예측
pred_df = pred_df.dropna(subset=x_cols)  # 결측치 있는 row 제거
pred_df['value'] = model.predict(pred_df[x_cols])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000986 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 8928, number of used features: 10
[LightGBM] [Info] Start training from score 6138446.366823


In [ ]:
submission = pred_df[['leader_item', 'follower_item', 'value']].copy()
submission.rename(columns={'leader_item': 'leading_item_id', 'follower_item': 'following_item_id'}, inplace=True)
submission['value'] = submission['value'].round().astype(int)
submission.head()

,leading_item_id,following_item_id,value
0,AANGBULD,APQGTRMF,36780
1,AANGBULD,DEWLVASR,345346
2,AANGBULD,DNMPSKTB,5918933
3,AANGBULD,EVBVXETX,5670737
4,AANGBULD,FTSVTTSR,213998


In [ ]:
submission.to_csv('submission1.csv', index=False)

pd.read_csv('submission1.csv')

,leading_item_id,following_item_id,value
0,AANGBULD,APQGTRMF,36780
1,AANGBULD,DEWLVASR,345346
2,AANGBULD,DNMPSKTB,5918933
3,AANGBULD,EVBVXETX,5670737
4,AANGBULD,FTSVTTSR,213998
...,...,...,...
1420,ZXERAXWP,DBWLZWNK,229066
1421,ZXERAXWP,FITUEHWN,123859
1422,ZXERAXWP,MIRCVAMV,50896
1423,ZXERAXWP,UIFPPCLR,94466
